# Driver Behavior Detection (YOLO11m + MediaPipe Pose) - Colab

Notebook này tối ưu cho bài toán khóa luận **nhận diện hành vi tài xế** với 4 class detector:

- `phone`
- `smoking`
- `seatbelt`
- `no-seatbelt`

## Mục tiêu của notebook
- **Train detector 4 class bằng YOLO11m**
- **Suy luận hành vi bằng MediaPipe Pose + rule-based engine**
- **Cài thư viện tối giản cho Colab**: không force-reinstall `torch`, `numpy`, `pillow`
- **Ổn định để resume train trên Google Drive**

## Ý tưởng tổng thể
1. **YOLO11m** học phát hiện các tín hiệu: phone / smoking / seatbelt / no-seatbelt  
2. **MediaPipe Pose** lấy landmark cơ thể tài xế: vai, tai, cổ tay, mũi, hông  
3. **Rule-based behavior engine** kết hợp:
   - khoảng cách từ box `phone` tới tay / tai
   - khoảng cách từ box `smoking` tới tay / miệng / mũi
   - `chest ROI` cho bài toán dây an toàn
   - smoothing theo thời gian cho video

## Khi nào dùng `last.pt` và `best.pt`
- **Resume train**: dùng `last.pt`
- **Inference / đánh giá**: ưu tiên `best.pt`


In [ ]:
# =========================
# 1) CÀI THƯ VIỆN TỐI GIẢN CHO COLAB
#    Mục tiêu: train YOLO ổn định, tránh xung đột torch/numpy/pillow/protobuf
# =========================
import os, sys, subprocess

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

print("Khuyên dùng: Runtime > Factory reset runtime trước khi chạy notebook lần đầu.")
print("Cell này KHÔNG cài MediaPipe để tránh lỗi protobuf / TensorFlow trên Colab.")


def sh(cmd: str):
    print(f"\n>>> {cmd}")
    subprocess.check_call(cmd, shell=True)


# Nâng pip
sh("python -m pip install -q --upgrade pip")

# Chỉ cài thư viện cần cho YOLO training + Roboflow dataset
sh('python -m pip install -q "ultralytics==8.3.32" "roboflow>=1.1.48,<2"')

# Kiểm tra môi trường hiện tại
import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image
import ultralytics

print("\n===== ENV CHECK =====")
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cv2:", cv2.__version__)
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nMôi trường train YOLO đã sẵn sàng.")

In [ ]:

# =========================
# 2) MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')



## Cấu hình dữ liệu

Notebook này dùng **một luồng dữ liệu duy nhất từ Roboflow**.

### Cách dùng
1. Tạo / kiểm tra version dataset trên Roboflow
2. Lấy **API Key** trong tài khoản Roboflow
3. Điền:
   - `ROBOFLOW_API_KEY`
   - `ROBOFLOW_WORKSPACE`
   - `ROBOFLOW_PROJECT`
   - `ROBOFLOW_VERSION`
4. Chạy cell tải dataset từ Roboflow

### Ghi chú
- Dataset sẽ được tải về thư mục local trong Colab: `/content/driver_behavior_dataset`
- File checkpoint vẫn lưu trên Google Drive để có thể resume train ở phiên khác
- Khi đổi tài khoản Google, bạn chỉ cần share thư mục Drive chứa `last.pt`


In [ ]:
# =========================
# 3) CẤU HÌNH CHÍNH
# =========================
from pathlib import Path
import os

# ===== ROBOFLOW DATASET =====
USE_ROBOFLOW_DOWNLOAD = True
ROBOFLOW_API_KEY = "fS2U662G0Hoxz1aJCzgh"
ROBOFLOW_WORKSPACE = "ladailoc-yzh0x"
ROBOFLOW_PROJECT = "phone-detect-svavs"
ROBOFLOW_VERSION = 7

# ===== THƯ MỤC LOCAL / DRIVE =====
LOCAL_DATASET_DIR = "/content/driver_behavior_dataset"
RUNS_DIR_IN_DRIVE = "/content/drive/MyDrive/driver_behavior_runs_train_100epochs"

# ===== TÊN RUN =====
EXP_NAME = "driver_behavior_yolo11m_mediapipe_rf"

# ===== RESUME =====
# Train mới: để None
# Train tiếp: điền thẳng đường dẫn last.pt trên Google Drive
# RESUME_CKPT = None
# Ví dụ:
RESUME_CKPT = "/content/drive/MyDrive/driver_behavior_runs_train_100epochs/driver_behavior_yolo11m_mediapipe_rf/weights/last.pt"

# ===== MODEL =====
MODEL_SIZE = "yolo11m.pt"

# ===== TRAINING =====
IMG_SIZE = 768
EPOCHS = 100
BATCH = 16
WORKERS = 2
PATIENCE = 20
CACHE = False
SAVE_PERIOD = -1
SEED = 42

# ===== DETECTOR INFERENCE =====
DET_CONF = 0.35

# ===== MEDIAPIPE =====
MP_MODEL_COMPLEXITY = 1
MP_MIN_DET_CONF = 0.45
MP_MIN_TRACK_CONF = 0.45
MP_MIN_VIS = 0.35
POSE_BRIGHTEN_GAMMA = 1.20   # tăng nhẹ sáng trước khi chạy pose nếu ảnh tối
USE_BRIGHTEN_FOR_POSE = True

Path(RUNS_DIR_IN_DRIVE).mkdir(parents=True, exist_ok=True)

print("RUNS_DIR_IN_DRIVE =", RUNS_DIR_IN_DRIVE)
print("EXP_NAME =", EXP_NAME)


In [ ]:

# =========================
# 4) TẢI DATASET TỪ ROBOFLOW
# =========================
from roboflow import Roboflow
from pathlib import Path
import shutil

assert USE_ROBOFLOW_DOWNLOAD is True, "Notebook bản này chỉ dùng dataset từ Roboflow."
assert ROBOFLOW_API_KEY.strip() != "", "Hãy điền ROBOFLOW_API_KEY trước khi chạy cell này."

local_dataset_dir = Path(LOCAL_DATASET_DIR)
if local_dataset_dir.exists():
    shutil.rmtree(local_dataset_dir)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Export dạng yolov8 để dùng trực tiếp với Ultralytics YOLO11
dataset = version.download("yolov8", location=str(local_dataset_dir))
dataset_dir_used = Path(LOCAL_DATASET_DIR)

print("Đã tải dataset từ Roboflow về:", dataset_dir_used)
print("Các file/thư mục chính:", [p.name for p in dataset_dir_used.iterdir()])


In [ ]:

# =========================
# 5) KIỂM TRA DATASET VỪA TẢI TỪ ROBOFLOW
# =========================
import yaml
from pathlib import Path

dataset_dir_used = Path(LOCAL_DATASET_DIR)
assert dataset_dir_used.exists(), f"Không thấy dataset ở: {dataset_dir_used}"
assert (dataset_dir_used / "data.yaml").exists(), "Thiếu file data.yaml sau khi tải từ Roboflow"

yaml_path = dataset_dir_used / "data.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("Nội dung data.yaml ban đầu:")
print(data_cfg)



## Chuẩn hóa tên class

Dataset của bạn hiện có các class như:
- `phone`
- `seatbelt`
- `no-seatbelt`
- `smoking`

Notebook sẽ chuẩn hóa alias để xử lý thống nhất:
- `no seatbelt`, `no_seatbelt`, `no-seatbelt` -> `no-seatbelt`
- `cell phone`, `mobile-phone` -> `phone`

Cell dưới đây chỉ sửa `data.yaml`, **không đổi file label index**.
Vì vậy **thứ tự class phải giữ nguyên** với dataset hiện tại.


In [ ]:

# =========================
# 6) CHUẨN HÓA data.yaml
# =========================
from pathlib import Path
import yaml

yaml_path = Path(dataset_dir_used) / "data.yaml"

def canonical_name(name: str) -> str:
    x = str(name).strip().lower().replace("_", "-")
    alias = {
        "cell-phone": "phone",
        "cell phone": "phone",
        "mobile-phone": "phone",
        "mobile phone": "phone",
        "no seatbelt": "no-seatbelt",
        "no-seat-belt": "no-seatbelt",
        "no_seatbelt": "no-seatbelt",
        "without-seatbelt": "no-seatbelt",
        "without seatbelt": "no-seatbelt",
        "seat-belt": "seatbelt",
        "cigarette": "smoking",
        "smoke": "smoking",
    }
    return alias.get(x, x)

with open(yaml_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

names_raw = cfg["names"]
if isinstance(names_raw, dict):
    names_list = [names_raw[k] for k in sorted(names_raw.keys())]
else:
    names_list = list(names_raw)

names_fixed = [canonical_name(n) for n in names_list]
cfg["names"] = names_fixed
cfg["nc"] = len(names_fixed)

# Chuẩn hóa path về local dataset
cfg["path"] = str(dataset_dir_used)
cfg["train"] = "train/images"
cfg["val"] = "valid/images" if (Path(dataset_dir_used) / "valid/images").exists() else "val/images"
cfg["test"] = "test/images" if (Path(dataset_dir_used) / "test/images").exists() else cfg.get("test", "test/images")

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("data.yaml sau chuẩn hóa:")
print(cfg)


In [ ]:
# =========================
# 7) KIỂM TRA NHANH DATASET + THỐNG KÊ CLASS
# =========================
from pathlib import Path
from collections import Counter
import yaml

yaml_path = Path(dataset_dir_used) / "data.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

names_raw = cfg["names"]
if isinstance(names_raw, dict):
    class_names = {k: names_raw[k] for k in sorted(names_raw.keys())}
elif isinstance(names_raw, list):
    class_names = {i: name for i, name in enumerate(names_raw)}
else:
    class_names = {}

num_classes = len(class_names)
print(f"📌 Đã tìm thấy {num_classes} classes trong data.yaml:")
print(class_names)
print("")

# Khởi tạo dict đếm số lượng: class_id -> {split: count}
class_counts = {c_name: {"train": 0, "valid": 0, "test": 0, "total": 0} for c_name in class_names.values()}

splits_mapped = {"train": "train", "valid": "valid", "val": "valid", "test": "test"}

for split_dir, split_name in splits_mapped.items():
    lbl_dir = Path(dataset_dir_used) / split_dir / "labels"
    if not lbl_dir.exists():
        continue
    
    for txt_path in lbl_dir.rglob("*.txt"):
        try:
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    if parts:
                        cls_id = int(float(parts[0]))
                        if cls_id in class_names:
                            c_name = class_names[cls_id]
                            class_counts[c_name][split_name] += 1
                            class_counts[c_name]["total"] += 1
        except Exception:
            pass

print("============================================================")
print(f"{'TÊN NHÃN (CLASS)':<20} | {'TRAIN':<8} | {'VALID':<8} | {'TEST':<8} | {'TỔNG CỘNG'}")
print("============================================================")

max_count = 0
min_count = float('inf')

for c_name in class_names.values():
    counts = class_counts[c_name]
    train_c = counts["train"]
    valid_c = counts["valid"]
    test_c = counts["test"]
    total_c = counts["total"]
    
    print(f"{c_name:<20} | {train_c:<8} | {valid_c:<8} | {test_c:<8} | {total_c:<8}")
    
    if total_c > max_count:
        max_count = total_c
    if total_c < min_count and total_c > 0:
        min_count = total_c

print("============================================================")
print("")

if min_count > 0 and min_count != float('inf'):
    ratio = max_count / min_count
    if ratio > 1.5:
        print(f"⚠️ CẢNH BÁO MẤT CÂN BẰNG: Class nhiều nhất gấp {ratio:.1f} lần class ít nhất.")

## Train / Resume Train

### Thiết kế train cho đồ án
- Train bằng **1 model multi-class** với 4 class detector:
  - `phone`
  - `smoking`
  - `seatbelt`
  - `no-seatbelt`
- Lưu **trực tiếp lên Google Drive** để:
  - tránh mất checkpoint khi Colab ngắt
  - dễ resume ở tài khoản Google khác
- Dùng `save_period=1` để luôn có `last.pt`
- Tắt W&B để tránh lỗi log/project name

### Lưu ý rất quan trọng về resume
- Nếu Colab bị ngắt giữa chừng, hãy resume bằng **`last.pt`**
- `best.pt` phù hợp để **suy luận / đánh giá**
- `last.pt` giữ trạng thái optimizer và epoch tốt hơn cho train tiếp


In [ ]:
# =========================
# 8) TRAIN / RESUME TRAIN
# =========================
import os
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO, settings

settings.update({"wandb": False})

print("numpy:", np.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

yaml_path = str(Path(dataset_dir_used) / "data.yaml")
save_dir_drive = Path(RUNS_DIR_IN_DRIVE) / EXP_NAME

if RESUME_CKPT is not None and str(RESUME_CKPT).strip() != "":
    assert Path(RESUME_CKPT).exists(), f"Không thấy checkpoint để resume: {RESUME_CKPT}"
    print("Resume từ checkpoint:", RESUME_CKPT)
    model = YOLO(RESUME_CKPT)
    results = model.train(
        data=yaml_path,
        resume=True
    )
else:
    print("Train mới từ base model:", MODEL_SIZE)
    model = YOLO(MODEL_SIZE)
    results = model.train(
        data=yaml_path,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        cache=CACHE,
        project=RUNS_DIR_IN_DRIVE,
        name=EXP_NAME,
        exist_ok=True,
        save=True,
        save_period=SAVE_PERIOD,
        amp=True,
        plots=True,

        cos_lr=True,
        close_mosaic=10,

        degrees=5.0,
        translate=0.1,
        scale=0.4,
        shear=0.0,
        perspective=0.0,

        mosaic=0.5,
        mixup=0.1,
        copy_paste=0.1,

        fliplr=0.5,
        flipud=0.0,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        optimizer="auto",
        seed=SEED,
        deterministic=True,
        verbose=True,
    )

print("Thư mục run:", save_dir_drive)
print("Nếu Colab ngắt, hãy resume từ file:", save_dir_drive / "weights" / "last.pt")

In [ ]:

# =========================
# 9) XÁC ĐỊNH best.pt / last.pt
# =========================
from pathlib import Path

exp_dir = Path(RUNS_DIR_IN_DRIVE) / EXP_NAME
weights_dir = exp_dir / "weights"

best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

print("best.pt exists:", best_pt.exists(), best_pt)
print("last.pt exists:", last_pt.exists(), last_pt)

MODEL_FOR_INFER = str(best_pt if best_pt.exists() else last_pt)
print("MODEL_FOR_INFER =", MODEL_FOR_INFER)


In [ ]:
# =========================
# 10) VAL / ĐÁNH GIÁ MODEL LÊN TẬP DATASET (THỐNG KÊ NHÃN & ACCURACY)
# =========================
from ultralytics import YOLO

model = YOLO(MODEL_FOR_INFER)
split_name = "test" if (Path(dataset_dir_used) / "test/images").exists() else "val"

print(f"Đang tiến hành đánh giá trên tập dữ liệu: {split_name.upper()}...")
metrics = model.val(
    data=str(Path(dataset_dir_used) / "data.yaml"),
    imgsz=IMG_SIZE,
    split=split_name
)

# Thống kê rõ ràng số lượng nhãn và kết quả Accuracy (Precision, Recall, mAP) cho từng class
print("\n=======================================================")
print("📊 BẢNG THỐNG KÊ ĐỘ CHÍNH XÁC (ACCURACY) TRÊN TỪNG NHÃN")
print("=======================================================")

class_indices = metrics.box.ap_class_index # Lấy index class có trong tập đánh giá
names = model.names

for i, cls_idx in enumerate(class_indices):
    cls_name = names[cls_idx]
    precision = metrics.box.p[i]    # Độ chuẩn xác
    recall = metrics.box.r[i]       # Độ phủ bao quát
    map50 = metrics.box.ap50[i]     # mAP ở IOU=0.5 (Dùng biến ap50 thay vì map50)
    map50_95 = metrics.box.ap[i]    # mAP ở IOU=0.5:0.95 (Dùng biến ap thay vì map)
    
    print(f"🔹 Nhãn: {cls_name.upper()}")
    print(f"   - Precision (Độ chính xác): {precision:.4f}")
    print(f"   - Recall (Độ phủ):          {recall:.4f}")
    print(f"   - mAP@50 (Accuracy 50):     {map50:.4f}")
    print(f"   - mAP@50-95 (Accuracy 95):  {map50_95:.4f}")
    print("-" * 55)

print("📈 TRUNG BÌNH TỔNG THỂ (OVERALL):")
print(f"   - Mean Precision: {metrics.box.mp:.4f}")    # Chỉ số chung dùng:.4f}") # Chỉ số chung dùng map50 / mp
print(f"   - Mean Recall:    {metrics.box.mr:.4f}")    # Mean recall
print(f"   - Mean mAP@50:    {metrics.box.map50:.4f}") # Mean mAP50
print(f"   - Mean mAP@50-95: {metrics.box.map:.4f}")   # Mean mAP50-95
print("=======================================================\n")

# Thiết kế thuật toán hành vi với MediaPipe

## 1) Sử dụng điện thoại (`using_phone`)
Không chỉ detect `phone` rồi kết luận ngay.

Luồng đề xuất:
- YOLO detector phát hiện box `phone`
- MediaPipe Pose lấy landmark tài xế:
  - cổ tay trái/phải
  - tai trái/phải
  - mũi
  - vai trái/phải
  - hông trái/phải
- Tạo **driver ROI** và **chest ROI**
- Chấm điểm `phone` theo:
  - gần tay
  - gần đầu / tai
  - nằm trong ROI của tài xế
  - kích thước hợp lý

## 2) Hút thuốc (`smoking`)
- Dùng box `smoking`
- Cộng điểm nếu box gần miệng / mũi / tay
- Loại bớt box quá xa vùng đầu và tay

## 3) Không thắt dây an toàn (`no_seatbelt`)
- Không nên chỉ nhìn vào conf của một box
- So sánh `seatbelt` và `no-seatbelt`
- Ưu tiên box giao với **chest ROI**
- Video dùng voting theo cửa sổ để tránh nhấp nháy


In [ ]:
# =========================
# 11) ENGINE SUY LUẬN HÀNH VI (YOLO + MEDIAPIPE)
# =========================
import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO

behavior_model = YOLO(MODEL_FOR_INFER)

CLASS_NAMES = behavior_model.names
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = {int(k): v for k, v in CLASS_NAMES.items()}
else:
    CLASS_NAMES = {i: n for i, n in enumerate(CLASS_NAMES)}


def cname(x: str) -> str:
    x = str(x).lower().replace("_", "-").strip()
    alias = {
        "no seatbelt": "no-seatbelt",
        "no_seatbelt": "no-seatbelt",
        "seat-belt": "seatbelt",
        "cell phone": "phone",
        "cell-phone": "phone",
        "mobile-phone": "phone",
        "mobile phone": "phone",
        "smoke": "smoking",
        "cigarette": "smoking",
    }
    return alias.get(x, x)


CLASS_NAMES = {k: cname(v) for k, v in CLASS_NAMES.items()}
print("CLASS_NAMES =", CLASS_NAMES)

mp_pose = mp.solutions.pose
LM = mp_pose.PoseLandmark

POSE_INDEX = {
    "nose": LM.NOSE,
    "left_ear": LM.LEFT_EAR,
    "right_ear": LM.RIGHT_EAR,
    "left_shoulder": LM.LEFT_SHOULDER,
    "right_shoulder": LM.RIGHT_SHOULDER,
    "left_elbow": LM.LEFT_ELBOW,
    "right_elbow": LM.RIGHT_ELBOW,
    "left_wrist": LM.LEFT_WRIST,
    "right_wrist": LM.RIGHT_WRIST,
    "left_hip": LM.LEFT_HIP,
    "right_hip": LM.RIGHT_HIP,
    "mouth_left": LM.MOUTH_LEFT,
    "mouth_right": LM.MOUTH_RIGHT,
}


def brighten_for_pose(img_bgr, gamma=1.20):
    if gamma is None or gamma <= 0 or abs(gamma - 1.0) < 1e-6:
        return img_bgr
    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img_bgr, table)


class MediaPipeDriverAnalyzer:
    def __init__(
        self,
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.45,
        min_tracking_confidence=0.45,
        min_visibility=0.35,
        gamma=1.20,
        use_brighten=True,
    ):
        self.pose = mp_pose.Pose(
            static_image_mode=static_image_mode,
            model_complexity=model_complexity,
            enable_segmentation=False,
            smooth_landmarks=not static_image_mode,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.min_visibility = min_visibility
        self.gamma = gamma
        self.use_brighten = use_brighten

    def close(self):
        self.pose.close()

    def extract_points(self, frame_bgr):
        work = frame_bgr
        if self.use_brighten:
            work = brighten_for_pose(work, self.gamma)

        rgb = cv2.cvtColor(work, cv2.COLOR_BGR2RGB)
        result = self.pose.process(rgb)
        if result.pose_landmarks is None:
            return None, result

        h, w = frame_bgr.shape[:2]
        pts = {}
        for name, idx in POSE_INDEX.items():
            lm = result.pose_landmarks.landmark[idx.value]
            if lm.visibility < self.min_visibility:
                pts[name] = None
            else:
                pts[name] = (lm.x * w, lm.y * h)
        return pts, result


def get_pt(pts, name):
    if pts is None:
        return None
    return pts.get(name)


def euclidean(p1, p2):
    return float(np.hypot(p1[0] - p2[0], p1[1] - p2[1]))


def midpoint(p1, p2):
    return ((p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0)


def clamp_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(w - 1, x1))
    y1 = max(0, min(h - 1, y1))
    x2 = max(0, min(w - 1, x2))
    y2 = max(0, min(h - 1, y2))
    return [float(x1), float(y1), float(x2), float(y2)]


def point_in_box(pt, box):
    if pt is None or box is None:
        return False
    x, y = pt
    x1, y1, x2, y2 = box
    return x1 <= x <= x2 and y1 <= y <= y2


def box_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def overlap_ratio(box_a, box_b):
    if box_a is None or box_b is None:
        return 0.0
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = box_area([ix1, iy1, ix2, iy2])
    return inter / max(1.0, box_area(box_a))


def get_shoulder_width(pts):
    ls = get_pt(pts, "left_shoulder")
    rs = get_pt(pts, "right_shoulder")
    if ls is not None and rs is not None:
        return max(1.0, euclidean(ls, rs))
    lh = get_pt(pts, "left_hip")
    rh = get_pt(pts, "right_hip")
    if lh is not None and rh is not None:
        return max(1.0, euclidean(lh, rh) * 0.9)
    return 80.0


def build_driver_roi(pts, w, h):
    if pts is None:
        return None
    valid = [p for p in pts.values() if p is not None]
    if len(valid) < 4:
        return None
    xs = [p[0] for p in valid]
    ys = [p[1] for p in valid]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)
    pad_x = (x2 - x1) * 0.30 + 20
    pad_y = (y2 - y1) * 0.25 + 20
    return clamp_box([x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y], w, h)


def build_chest_roi(pts, w, h):
    if pts is None:
        return None
    ls = get_pt(pts, "left_shoulder")
    rs = get_pt(pts, "right_shoulder")
    lh = get_pt(pts, "left_hip")
    rh = get_pt(pts, "right_hip")
    if ls is None or rs is None:
        return None
    shoulder_width = get_shoulder_width(pts)
    shoulder_mid = midpoint(ls, rs)

    if lh is not None and rh is not None:
        hip_mid = midpoint(lh, rh)
        chest_h = abs(hip_mid[1] - shoulder_mid[1]) * 0.65
    else:
        chest_h = shoulder_width * 0.95

    x1 = shoulder_mid[0] - shoulder_width * 0.70
    x2 = shoulder_mid[0] + shoulder_width * 0.70
    y1 = shoulder_mid[1] - shoulder_width * 0.18
    y2 = y1 + chest_h
    return clamp_box([x1, y1, x2, y2], w, h)


def parse_det_result(det_result):
    boxes = []
    if det_result.boxes is None:
        return boxes

    xyxy = det_result.boxes.xyxy.cpu().numpy()
    confs = det_result.boxes.conf.cpu().numpy()
    clss = det_result.boxes.cls.cpu().numpy().astype(int)

    for box, conf, cls_id in zip(xyxy, confs, clss):
        class_name = CLASS_NAMES.get(int(cls_id), str(cls_id))
        x1, y1, x2, y2 = map(float, box.tolist())
        boxes.append({
            "xyxy": [x1, y1, x2, y2],
            "conf": float(conf),
            "cls_id": int(cls_id),
            "class_name": class_name,
            "center": ((x1 + x2) / 2.0, (y1 + y2) / 2.0),
            "area": box_area([x1, y1, x2, y2]),
        })
    return boxes


def score_phone_behavior(phone_box, pts, driver_roi=None):
    score = phone_box["conf"]
    shoulder_width = get_shoulder_width(pts)
    c = phone_box["center"]

    nose = get_pt(pts, "nose")
    l_ear = get_pt(pts, "left_ear")
    r_ear = get_pt(pts, "right_ear")
    l_wrist = get_pt(pts, "left_wrist")
    r_wrist = get_pt(pts, "right_wrist")

    head_pts = [p for p in [nose, l_ear, r_ear] if p is not None]
    hand_pts = [p for p in [l_wrist, r_wrist] if p is not None]

    if hand_pts:
        d_hand = min(euclidean(c, p) for p in hand_pts) / shoulder_width
        if d_hand < 1.10:
            score += 0.28
        elif d_hand < 1.45:
            score += 0.12

    if head_pts:
        d_head = min(euclidean(c, p) for p in head_pts) / shoulder_width
        if d_head < 1.00:
            score += 0.20
        elif d_head < 1.35:
            score += 0.08

    if driver_roi is not None and point_in_box(c, driver_roi):
        score += 0.15

    rel_area = phone_box["area"] / max(1.0, shoulder_width * shoulder_width)
    if 0.01 <= rel_area <= 0.22:
        score += 0.08
    else:
        score -= 0.12

    return float(score)


def score_smoking_behavior(smoke_box, pts, driver_roi=None):
    score = smoke_box["conf"]
    shoulder_width = get_shoulder_width(pts)
    c = smoke_box["center"]

    nose = get_pt(pts, "nose")
    mouth_l = get_pt(pts, "mouth_left")
    mouth_r = get_pt(pts, "mouth_right")
    l_wrist = get_pt(pts, "left_wrist")
    r_wrist = get_pt(pts, "right_wrist")

    face_pts = [p for p in [nose, mouth_l, mouth_r] if p is not None]
    hand_pts = [p for p in [l_wrist, r_wrist] if p is not None]

    if face_pts:
        d_face = min(euclidean(c, p) for p in face_pts) / shoulder_width
        if d_face < 0.95:
            score += 0.25
        elif d_face < 1.25:
            score += 0.10

    if hand_pts:
        d_hand = min(euclidean(c, p) for p in hand_pts) / shoulder_width
        if d_hand < 1.00:
            score += 0.18
        elif d_hand < 1.35:
            score += 0.08

    if driver_roi is not None and point_in_box(c, driver_roi):
        score += 0.10

    rel_area = smoke_box["area"] / max(1.0, shoulder_width * shoulder_width)
    if 0.002 <= rel_area <= 0.10:
        score += 0.08
    else:
        score -= 0.10

    return float(score)


def resolve_seatbelt_state(boxes, chest_roi=None):
    seatbelt_boxes = [b for b in boxes if b["class_name"] == "seatbelt"]
    no_seatbelt_boxes = [b for b in boxes if b["class_name"] == "no-seatbelt"]

    def best_score(cands):
        best = 0.0
        for b in cands:
            s = b["conf"]
            if chest_roi is not None:
                s += 0.20 * overlap_ratio(b["xyxy"], chest_roi)
            best = max(best, s)
        return best

    seatbelt_best = best_score(seatbelt_boxes)
    no_seatbelt_best = best_score(no_seatbelt_boxes)

    if no_seatbelt_best >= 0.45 and no_seatbelt_best > seatbelt_best + 0.07:
        return "no_seatbelt", no_seatbelt_best
    if seatbelt_best >= 0.45 and seatbelt_best > no_seatbelt_best + 0.07:
        return "seatbelt", seatbelt_best
    if max(seatbelt_best, no_seatbelt_best) >= 0.50:
        return "uncertain", max(seatbelt_best, no_seatbelt_best)
    return "unknown", max(seatbelt_best, no_seatbelt_best)


def make_pose_analyzer(static_image_mode=False):
    return MediaPipeDriverAnalyzer(
        static_image_mode=static_image_mode,
        model_complexity=MP_MODEL_COMPLEXITY,
        min_detection_confidence=MP_MIN_DET_CONF,
        min_tracking_confidence=MP_MIN_TRACK_CONF,
        min_visibility=MP_MIN_VIS,
        gamma=POSE_BRIGHTEN_GAMMA,
        use_brighten=USE_BRIGHTEN_FOR_POSE,
    )


def infer_behaviors(frame_bgr, pose_analyzer=None):
    h, w = frame_bgr.shape[:2]

    det_result = behavior_model.predict(frame_bgr, imgsz=IMG_SIZE, conf=DET_CONF, verbose=False)[0]
    boxes = parse_det_result(det_result)

    owns_pose = pose_analyzer is None
    if pose_analyzer is None:
        pose_analyzer = make_pose_analyzer(static_image_mode=True)

    pts, pose_result = pose_analyzer.extract_points(frame_bgr)
    driver_roi = build_driver_roi(pts, w, h)
    chest_roi = build_chest_roi(pts, w, h)

    behaviors = {
        "using_phone": {"active": False, "score": 0.0},
        "smoking": {"active": False, "score": 0.0},
        "seatbelt_state": {"label": "unknown", "score": 0.0},
    }

    if pts is not None:
        phone_boxes = [b for b in boxes if b["class_name"] == "phone"]
        smoke_boxes = [b for b in boxes if b["class_name"] == "smoking"]

        if phone_boxes:
            phone_scores = [score_phone_behavior(b, pts, driver_roi) for b in phone_boxes]
            best_phone_score = max(phone_scores)
            behaviors["using_phone"]["score"] = best_phone_score
            behaviors["using_phone"]["active"] = best_phone_score >= 0.58

        if smoke_boxes:
            smoke_scores = [score_smoking_behavior(b, pts, driver_roi) for b in smoke_boxes]
            best_smoke_score = max(smoke_scores)
            behaviors["smoking"]["score"] = best_smoke_score
            behaviors["smoking"]["active"] = best_smoke_score >= 0.58

    seatbelt_label, seatbelt_score = resolve_seatbelt_state(boxes, chest_roi=chest_roi)
    behaviors["seatbelt_state"] = {"label": seatbelt_label, "score": seatbelt_score}

    if owns_pose:
        pose_analyzer.close()

    return boxes, behaviors, driver_roi, chest_roi, pts


print("Đã sẵn sàng infer behaviors bằng YOLO detector + MediaPipe Pose.")


In [ ]:
# =========================
# 12) TEST NHANH TRÊN 1 ẢNH
# =========================
from matplotlib import pyplot as plt

IMAGE_PATH = None
# Ví dụ:
# IMAGE_PATH = "/content/drive/MyDrive/test_driver.jpg"

if IMAGE_PATH is None:
    print("Hãy gán IMAGE_PATH rồi chạy lại cell.")
else:
    img = cv2.imread(IMAGE_PATH)
    assert img is not None, f"Không đọc được ảnh: {IMAGE_PATH}"

    pose_analyzer = make_pose_analyzer(static_image_mode=True)
    boxes, behaviors, driver_roi, chest_roi, pts = infer_behaviors(img.copy(), pose_analyzer=pose_analyzer)
    pose_analyzer.close()

    vis = img.copy()

    for b in boxes:
        x1, y1, x2, y2 = map(int, b["xyxy"])
        label = f'{b["class_name"]} {b["conf"]:.2f}'
        color = (0, 255, 0)
        if b["class_name"] == "no-seatbelt":
            color = (0, 0, 255)
        elif b["class_name"] == "smoking":
            color = (255, 140, 0)
        elif b["class_name"] == "phone":
            color = (255, 0, 255)

        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if driver_roi is not None:
        x1, y1, x2, y2 = map(int, driver_roi)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 200, 0), 2)
        cv2.putText(vis, "driver_roi", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)

    if chest_roi is not None:
        x1, y1, x2, y2 = map(int, chest_roi)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(vis, "chest_roi", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    if pts is not None:
        for name, pt in pts.items():
            if pt is None:
                continue
            x, y = map(int, pt)
            cv2.circle(vis, (x, y), 4, (255, 255, 0), -1)

    lines = [
        f'using_phone: {behaviors["using_phone"]["active"]} ({behaviors["using_phone"]["score"]:.2f})',
        f'smoking: {behaviors["smoking"]["active"]} ({behaviors["smoking"]["score"]:.2f})',
        f'seatbelt_state: {behaviors["seatbelt_state"]["label"]} ({behaviors["seatbelt_state"]["score"]:.2f})',
    ]

    y0 = 30
    for i, t in enumerate(lines):
        cv2.putText(vis, t, (20, y0 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


## Temporal smoothing cho video

Nếu kết luận hành vi ngay ở từng frame thì kết quả sẽ dễ nhấp nháy.  
Vì vậy ta dùng:

- **EMA score**: làm mượt điểm tin cậy
- **K-frame confirmation**: chỉ bật hành vi nếu đủ số frame liên tiếp
- **majority vote** cho `seatbelt_state`
- riêng `no_seatbelt` cần voting theo cửa sổ để tránh frame lỗi

MediaPipe chạy khá nhẹ trên Colab cho suy luận video, nhất là khi chỉ dùng 1 người chính là tài xế.


In [ ]:
# =========================
# 13) SUY LUẬN VIDEO + SMOOTHING
# =========================
import cv2
from collections import deque, Counter

VIDEO_PATH = None
# Ví dụ:
# VIDEO_PATH = "/content/drive/MyDrive/test_driver_video.mp4"

OUTPUT_VIDEO_PATH = "/content/drive/MyDrive/driver_behavior_output.mp4"

EMA_ALPHA = 0.35
PHONE_ON_THRES = 0.58
SMOKE_ON_THRES = 0.58
MIN_CONSEC_FRAMES = 3
SEATBELT_WINDOW = 8

if VIDEO_PATH is None:
    print("Hãy gán VIDEO_PATH rồi chạy lại cell.")
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    assert cap.isOpened(), f"Không mở được video: {VIDEO_PATH}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps if fps > 0 else 25, (w, h))

    ema_phone = 0.0
    ema_smoke = 0.0
    phone_on_count = 0
    smoke_on_count = 0
    seatbelt_hist = deque(maxlen=SEATBELT_WINDOW)

    pose_analyzer = make_pose_analyzer(static_image_mode=False)

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        boxes, behaviors, driver_roi, chest_roi, pts = infer_behaviors(frame, pose_analyzer=pose_analyzer)

        ema_phone = EMA_ALPHA * behaviors["using_phone"]["score"] + (1 - EMA_ALPHA) * ema_phone
        ema_smoke = EMA_ALPHA * behaviors["smoking"]["score"] + (1 - EMA_ALPHA) * ema_smoke

        phone_on_count = phone_on_count + 1 if ema_phone >= PHONE_ON_THRES else 0
        smoke_on_count = smoke_on_count + 1 if ema_smoke >= SMOKE_ON_THRES else 0

        using_phone = phone_on_count >= MIN_CONSEC_FRAMES
        smoking = smoke_on_count >= MIN_CONSEC_FRAMES

        seatbelt_hist.append(behaviors["seatbelt_state"]["label"])
        seatbelt_mode = Counter(seatbelt_hist).most_common(1)[0][0] if len(seatbelt_hist) else "unknown"

        vis = frame.copy()

        for b in boxes:
            x1, y1, x2, y2 = map(int, b["xyxy"])
            label = f'{b["class_name"]} {b["conf"]:.2f}'
            color = (0, 255, 0)
            if b["class_name"] == "no-seatbelt":
                color = (0, 0, 255)
            elif b["class_name"] == "smoking":
                color = (255, 140, 0)
            elif b["class_name"] == "phone":
                color = (255, 0, 255)

            cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
            cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        if driver_roi is not None:
            x1, y1, x2, y2 = map(int, driver_roi)
            cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 200, 0), 2)

        if chest_roi is not None:
            x1, y1, x2, y2 = map(int, chest_roi)
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)

        if pts is not None:
            for pt in pts.values():
                if pt is None:
                    continue
                cv2.circle(vis, tuple(map(int, pt)), 3, (255, 255, 0), -1)

        panel = [
            f"using_phone: {using_phone} | ema={ema_phone:.2f}",
            f"smoking: {smoking} | ema={ema_smoke:.2f}",
            f"seatbelt_state: {seatbelt_mode}",
            f"frame: {frame_idx}",
        ]

        for i, txt in enumerate(panel):
            cv2.putText(vis, txt, (20, 35 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        out.write(vis)
        frame_idx += 1

    pose_analyzer.close()
    cap.release()
    out.release()
    print("Đã lưu video kết quả tại:", OUTPUT_VIDEO_PATH)


# Resume train trên tài khoản Google khác

Khi Colab hết GPU miễn phí, bạn có thể train tiếp như sau:

## Cách an toàn nhất
1. Vì notebook này lưu run **trực tiếp lên Google Drive**, sau mỗi epoch bạn đã có:
   - `best.pt`
   - `last.pt`
   - `results.csv`
   - `args.yaml`

2. Ở tài khoản A:
   - mở thư mục `driver_behavior_runs`
   - share thư mục run cho tài khoản B  
   hoặc copy nguyên thư mục run sang Drive của tài khoản B

3. Ở tài khoản B:
   - mount Google Drive
   - mở notebook này
   - gán:
```python
RESUME_CKPT = "/content/drive/MyDrive/driver_behavior_runs/driver_behavior_yolo11m_multibehavior_rf_v2/weights/last.pt"
```

4. Chạy lại cell **Train / Resume Train**

## Nguyên tắc chọn file
- `last.pt` -> dùng để **train tiếp**
- `best.pt` -> dùng để **suy luận / đánh giá**


# Gợi ý viết vào khóa luận

## Kiến trúc đề xuất
- **YOLO11m multi-class** phát hiện các tín hiệu hành vi: `phone`, `smoking`, `seatbelt`, `no-seatbelt`
- **MediaPipe Pose** cung cấp landmark cơ thể tài xế
- **Rule-based behavior engine** kết hợp:
  - vị trí box
  - khoảng cách tới tay / đầu / miệng
  - chest ROI cho bài toán dây an toàn
  - temporal smoothing trên video
- **Decision layer** sinh ra hành vi cuối:
  - `using_phone`
  - `smoking`
  - `no_seatbelt`

## Điểm nhấn có thể nêu trong báo cáo
1. **Detector và behavior tách rời**: model chỉ học detect object, còn behavior được suy ra bằng pose + luật.
2. **Dễ mở rộng**: sau này có thể thêm `drinking`, `yawning`, `looking_away` mà không cần đổi toàn bộ pipeline.
3. **Ổn định trên Colab**: cell cài thư viện ngắn, không phá torch / numpy / pillow.
4. **Phù hợp dữ liệu thực tế**: giảm false positive kiểu box điện thoại nằm ngoài vùng tài xế.
